# JPEG-DCT Classification, From First Principles

This notebook explains *why* the pieces look the way they do:

| | |
|---|---|
| 1 | Where the dataset comes from - a written account, no download |
| 2 | The problem: unfiltered images, polluted with people |
| 3 | The same classes after curation |
| 4 | What the DCT coefficients actually are |
| 5 | Rebuilding the image from *n* coefficients - what gets thrown away |
| 6 | Why DCT and not DFT - a 1-D demonstration |
| 7 | Building the network from scratch |
| 8 | Model summary |
| 9 | Splitting the data and training |
| 10 | Quantization-aware training - what and why |
| 11 | QAT implementation |
| 12 | int8 quantization |
| 13 | Weight export and verification |

> **Demo only.** Everything written goes to `youtube/output/first_principles/`. No project
> file is modified. The dataset and the project's library are read, never written.

In [ ]:
"""Setup output and write guard which raises if writing to a restricted location."""
import json, math, os, random, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def find_project(start=None):
    here = Path(start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "python_code" / "dct_common").is_dir():
            return cand
    raise RuntimeError("Could not find the project root above " + str(here))

PROJECT     = find_project()
NB_DIR      = PROJECT / "youtube"
PYTHON_CODE = PROJECT / "python_code"
DATA_DIR    = PROJECT / "data"                          # read-only
SOURCE_DIR  = PROJECT / "everyday_openimages160x120"    # read-only
NB_OUT      = NB_DIR / "output" / "first_principles"    # the only writable location
NB_OUT.mkdir(parents=True, exist_ok=True)

def ensure_import_path():
    if str(PYTHON_CODE) not in sys.path:
        sys.path.insert(0, str(PYTHON_CODE))

ensure_import_path()

def guard(path):
    p = Path(path).resolve()
    if NB_OUT.resolve() not in p.parents and p != NB_OUT.resolve():
        raise RuntimeError(f"BLOCKED: {p} is outside {NB_OUT}")
    return p

import dct_common
print(f"project    : {PROJECT}")
print(f"library    : {Path(dct_common.__file__).parent}")
print(f"writing to : {NB_OUT.relative_to(PROJECT)}/")

CLASSES = ["people", "computer", "doors", "fruit", "car"]
DEMO_MODE = True
FLOAT_EPOCHS, QAT_EPOCHS = (12, 5) if DEMO_MODE else (60, 20)
MAX_IMAGES_PER_CLASS = 260 if DEMO_MODE else None

---
## 1. Where the dataset comes from

Not run here - the download alone takes hours. This is the account of it.

**Source.** [Open Images V7](https://storage.googleapis.com/openimages/web/index.html),
pulled with [FiftyOne](https://docs.voxel51.com/). Open Images is a *detection* dataset:
every image carries bounding boxes for the objects annotators were asked to mark. We ask
for images containing particular labels - `Person`, `Laptop`, `Computer keyboard`, `Door`,
`Fruit`, `Car` - and take up to 1,000 per class.

```python
dataset = foz.load_zoo_dataset(
    "open-images-v7", split=oi_split,
    label_types=["detections"],
    classes=oi_classes,          # e.g. ["Laptop", "Computer keyboard"]
    max_samples=download_count, shuffle=True, seed=RANDOM_SEED,
)
```

**Resize and re-encode.** Each image is resized - never cropped - to **160x120**, the
OV2640's native `FRAMESIZE_QQVGA`, and re-encoded as JPEG with **4:2:2** chroma
subsampling to match what the real camera emits. This matters more than it sounds: the
model reads quantized coefficients straight out of the bitstream, so the training images
have to be JPEGs of the same geometry the camera will produce.

**Record what else is in the frame.** For every saved image the pipeline records which
*other* target classes were also annotated in the source photo, into
`meta/train_intersections.json`. Nothing is dropped at download time - filtering stays a
training-time decision.

**Run a detector over the saved pixels.** Open Images' annotations are incomplete: they
box salient objects, not every person in every photo. So `detect_people_yolo.py` runs
**YOLO11m** over the actual saved images and records the maximum person-detection
confidence per image, into `meta/train_yolo_people.json`. Again, the raw signal is stored,
not a verdict.

**Merge, filter, split.** `build_data.py` then merges the fine-grained source classes into
the deployed taxonomy (`keyboard` + `laptop` + `monitor` + `mouse` -> `computer`), drops
images caught by either people filter, and writes `data/{train,val,test}/`.

The scripts, in order: `get_everyday_openimages_data.py` -> `detect_people_yolo.py` ->
`build_data.py`.

In [ ]:
"""What that produced, on disk."""
src_classes = sorted(p.name for p in (SOURCE_DIR / "train").iterdir() if p.is_dir())
print(f"raw source classes ({len(src_classes)}):")
print("   ", ", ".join(src_classes))
print()
print(f"{'':<14}{'raw train':>11}{'raw val':>9}   |{'curated train':>15}{'val':>7}{'test':>7}")
print("-" * 68)
for c in CLASSES:
    raw_tr = len(list((SOURCE_DIR / "train" / c).glob("*.jpg"))) if (SOURCE_DIR / "train" / c).exists() else 0
    raw_va = len(list((SOURCE_DIR / "val" / c).glob("*.jpg"))) if (SOURCE_DIR / "val" / c).exists() else 0
    cur = [len(list((DATA_DIR / s / c).glob("*.jpg"))) for s in ("train", "val", "test")]
    note = "  (merged)" if raw_tr == 0 else ""
    print(f"{c:<14}{raw_tr:>11}{raw_va:>9}   |{cur[0]:>15}{cur[1]:>7}{cur[2]:>7}{note}")
print()
print("`computer` has no raw folder because it is a merge of keyboard/laptop/monitor/mouse.")

---
## 2. The problem, in pictures

Here are images the raw download labelled as furniture, doors, tables and so on - each of
which **also contains a person**. Train on these as-is and the network learns that people
are part of what a door looks like.

Two independent signals catch them: Open Images' own annotations, and a YOLO11m pass over
the saved pixels. The second exists because the first is incomplete.

In [ ]:
"""Show me the polluted ones."""
inter = json.load(open(SOURCE_DIR / "meta" / "train_intersections.json"))
yolo  = json.load(open(SOURCE_DIR / "meta" / "train_yolo_people.json"))
YOLO_CONF = 0.25     # matches filter_intersections.py

def show_grid(items, ncols=6, scale=2.1, suptitle=None):
    """items: list of (path, title, colour)."""
    items = [(p, t, c) for p, t, c in items if Path(p).exists()]
    nrows = (len(items) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * scale, nrows * scale * 1.18))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (p, t, col) in zip(axes, items):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(t, fontsize=8, color=col)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    fig.tight_layout(); plt.show()

polluted, seen = [], set()
for cls, files in sorted(inter.items()):
    if cls == "people":
        continue
    for fname, others in files.items():
        conf = yolo.get(cls, {}).get(fname, 0.0)
        by_annotation = "people" in (others or [])
        by_yolo = conf >= YOLO_CONF
        if not (by_annotation or by_yolo) or cls in seen:
            continue
        why = "annotation" if by_annotation and not by_yolo else ("YOLO %.2f" % conf if not by_annotation else "both")
        p = SOURCE_DIR / "train" / cls / fname
        if p.exists():
            polluted.append((p, f"{cls}\ncaught by: {why}", "#c02626")); seen.add(cls)
        break
    if len(polluted) >= 12:
        break

show_grid(polluted, suptitle="UNFILTERED: labelled as something else, but a person is in frame")

n_ann = sum(1 for cls, f in inter.items() if cls != "people" for _, o in f.items() if "people" in (o or []))
n_yolo_only = sum(1 for cls, f in yolo.items() if cls != "people" for fn, cf in f.items()
                  if cf >= YOLO_CONF and "people" not in (inter.get(cls, {}).get(fn) or []))
print(f"caught by Open Images annotations : {n_ann:,}")
print(f"caught ONLY by the YOLO pass      : {n_yolo_only:,}")
print(f"total removed                     : {n_ann + n_yolo_only:,}")

---
## 3. After curation

The same five classes, drawn at random from `data/` - what actually reaches the network.

In [ ]:
"""A random sample of the clean training set."""
rng = random.Random(4)
clean = []
for c in CLASSES:
    pool = sorted((DATA_DIR / "train" / c).glob("*.jpg"))
    for p in rng.sample(pool, min(4, len(pool))):
        clean.append((p, c, "#1a7f37"))

show_grid(clean, ncols=10, scale=1.7, suptitle="CURATED: the training set after merging and both people filters")

---
## 4. What the network actually reads

JPEG splits the image into 8x8 blocks and runs a 2-D DCT on each one, turning 64 pixels
into 64 frequency coefficients. Coefficient 0 - the **DC** term - is the block's mean
brightness. The rest describe increasingly fine detail, and are stored in **zig-zag
order**, lowest frequency first.

We never undo that. We read the quantized coefficients straight from the bitstream,
dequantize them with the image's own table, and keep the first few per block. At 160x120
with 4:2:2 that is a **15x20 grid** of blocks instead of a 120x160 grid of pixels.

In [ ]:
"""The coefficient planes for one image."""
from dct_common.config import Config
from dct_common.features import extract_y_dct_planes, extract_cbcr_dct_planes

cfg = Config(
    capture_width=160, capture_height=120, chroma_subsampling="4:2:2",
    num_ac_coeffs=3, num_chroma_ac_coeffs=0, use_chroma=True,
    lum_channels=16, stride2_channels=32, post_concat_channels=64,
    extra_conv_channels=(32,), selected_classes=tuple(CLASSES),
    epochs=FLOAT_EPOCHS, dropout=0.3, seed=1234,
)

demo_img = sorted((DATA_DIR / "test" / "fruit").glob("*.jpg"))[0]
Yp = extract_y_dct_planes(demo_img, cfg)
Cp = extract_cbcr_dct_planes(demo_img, cfg)

n_panels = 2 + cfg.num_ac_coeffs + 2
fig, axes = plt.subplots(1, n_panels, figsize=(2.9 * n_panels, 3.2))
axes[0].imshow(Image.open(demo_img).convert("RGB")); axes[0].set_title("original\n120x160 px", fontsize=9)
axes[1].imshow(Yp[0], cmap="gray"); axes[1].set_title(f"luma DC\n{cfg.y_rows}x{cfg.y_cols}", fontsize=9)
for k in range(cfg.num_ac_coeffs):
    axes[2 + k].imshow(Yp[1 + k], cmap="RdBu_r"); axes[2 + k].set_title(f"luma AC {k + 1}", fontsize=9)
axes[-2].imshow(Cp[0], cmap="PuOr"); axes[-2].set_title("Cb DC", fontsize=9)
axes[-1].imshow(Cp[cfg.num_chroma_coeffs], cmap="PuOr"); axes[-1].set_title("Cr DC", fontsize=9)
for ax in axes:
    ax.axis("off")
fig.suptitle("The DC plane is a thumbnail the camera has already computed for us", fontsize=11)
fig.tight_layout(); plt.show()

print(f"kept per frame: {cfg.num_coeffs * cfg.y_rows * cfg.y_cols + 2 * cfg.num_chroma_coeffs * cfg.c_rows * cfg.c_cols:,} values")
print(f"RGB equivalent: {3 * cfg.capture_width * cfg.capture_height:,} values")

---
## 5. Putting the image back together

The obvious question: if we keep 3 coefficients out of 64, what have we thrown away?

Here we invert the DCT ourselves. For each 8x8 block we dequantize, **zero everything past
the first *n* zig-zag positions**, and run an inverse DCT back to pixels. `n = 1` is the DC
term alone - every block becomes a flat square of its mean brightness, which is exactly
the thumbnail seen above. `n = 64` is the full JPEG.

The deployed model uses **n = 3**.

In [ ]:
"""Inverse DCT, keeping only the first n zig-zag coefficients per block."""
import jpeglib
from scipy.fft import idctn
from dct_common.zigzag import generate_zigzag_order

ZIGZAG = generate_zigzag_order(8)          # [(row, col), ...] in scan order

def reconstruct_luma(jpg_path, n_coeffs):
    """Decode the Y channel using only the first n_coeffs zig-zag positions."""
    im = jpeglib.read_dct(str(jpg_path))
    coeffs = im.Y.astype(np.float32) * im.qt[0].astype(np.float32)[None, None, :, :]

    keep = np.zeros((8, 8), dtype=np.float32)
    for r, c in ZIGZAG[:n_coeffs]:
        keep[r, c] = 1.0
    blocks = idctn(coeffs * keep, axes=(2, 3), type=2, norm="ortho") + 128.0

    bh, bw = blocks.shape[0], blocks.shape[1]
    return np.clip(blocks.transpose(0, 2, 1, 3).reshape(bh * 8, bw * 8), 0, 255).astype(np.uint8)

full = reconstruct_luma(demo_img, 64).astype(np.float32)
ns = [1, 2, 3, 6, 10, 21, 64]

fig, axes = plt.subplots(1, len(ns) + 1, figsize=(2.5 * (len(ns) + 1), 3.2))
axes[0].imshow(Image.open(demo_img).convert("RGB")); axes[0].set_title("original (colour)", fontsize=9)
for ax, n in zip(axes[1:], ns):
    rec = reconstruct_luma(demo_img, n)
    rmse = float(np.sqrt(np.mean((rec.astype(np.float32) - full) ** 2)))
    ax.imshow(rec, cmap="gray", vmin=0, vmax=255)
    label = "DC only" if n == 1 else ("all 64" if n == 64 else f"{n} coeffs")
    ax.set_title(f"{label}\nRMSE {rmse:.1f}", fontsize=9,
                 color="#c05621" if n == cfg.num_coeffs else "black")
for ax in axes:
    ax.axis("off")
fig.suptitle("Luma reconstructed from the first n zig-zag coefficients per 8x8 block "
             f"(orange = the {cfg.num_coeffs} the model keeps)", fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
"""How quickly does the error fall? This is the energy-compaction property."""
errs = []
ks = list(range(1, 65))
for n in ks:
    rec = reconstruct_luma(demo_img, n).astype(np.float32)
    errs.append(float(np.sqrt(np.mean((rec - full) ** 2))))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(ks, errs, color="#1f6f8b", lw=2)
ax.axvline(cfg.num_coeffs, color="#c05621", ls="--",
           label=f"deployed model keeps {cfg.num_coeffs}")
ax.set_xlabel("zig-zag coefficients retained per block"); ax.set_ylabel("RMSE vs full decode")
ax.set_title("Nearly all the energy is in the first few coefficients")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

print(f"RMSE at  1 coefficient : {errs[0]:.1f}")
print(f"RMSE at  {cfg.num_coeffs} coefficients: {errs[cfg.num_coeffs - 1]:.1f}")
print(f"RMSE at 21 coefficients: {errs[20]:.1f}")
print()
print("Worth being honest about: 3 coefficients is a POOR reconstruction. The image is")
print("visibly blocky and the RMSE has barely moved from DC-only. Yet those same 3")
print("coefficients classify at 81% on the five-class task.")
print()
print("Reconstruction quality and classification usefulness are not the same thing. The")
print("network is not trying to see the picture -- it needs whatever separates a door from")
print("a car, and coarse layout plus colour does most of that work. A coefficient sweep on")
print("this dataset found that going beyond DC bought training accuracy but not test")
print("accuracy, which says the extra planes were carrying mostly noise on THIS data.")

---
## 6. Why the DCT and not the DFT

Both transforms turn a signal into frequencies. JPEG uses the DCT, and the reason is worth
seeing rather than asserting.

A finite signal has to be *extended* to be treated as periodic. The DFT repeats it
end-to-end, so if the signal starts and ends at different values there is a **jump** at
every join. The DCT mirrors it first, so the extension is continuous.

A jump is expensive in frequency terms - it needs many coefficients to represent, and
truncating them produces ringing (the Gibbs phenomenon). No jump, no ringing, and the
energy concentrates in far fewer coefficients.

Take a signal that is deliberately awkward for the DFT: $y = \sqrt{x}$ on $[0, \pi]$,
which starts at 0 and ends at $\sqrt{\pi} \approx 1.77$.

In [ ]:
"""The two implied extensions, drawn out to 6*pi."""
from scipy.fft import dct, idct

N = 256
x = np.linspace(0, np.pi, N, endpoint=False)
y = np.sqrt(x)

reps = 3
x_ext = np.linspace(0, 6 * np.pi, reps * 2 * N, endpoint=False)
dft_ext = np.tile(y, reps * 2)                      # periodic repeat, period pi
dct_ext = np.tile(np.concatenate([y, y[::-1]]), reps)   # mirror first, period 2*pi

fig, axes = plt.subplots(2, 1, figsize=(11, 5.2), sharex=True)
axes[0].plot(x_ext, dft_ext, color="#c05621", lw=1.6)
axes[0].set_title("DFT's implied extension: repeat -- a jump at every period boundary", fontsize=11)
axes[1].plot(x_ext, dct_ext, color="#1f6f8b", lw=1.6)
axes[1].set_title("DCT's implied extension: mirror, then repeat -- continuous everywhere", fontsize=11)
for ax in axes:
    for k in range(1, reps * 2):
        ax.axvline(k * np.pi, color="grey", ls=":", alpha=0.5)
    ax.grid(alpha=0.25); ax.set_ylabel("y")
axes[1].set_xlabel("x")
axes[1].set_xticks([k * np.pi for k in range(7)],
                   ["0"] + [f"{k}$\\pi$" for k in range(1, 7)])
fig.suptitle(r"$y=\sqrt{x}$ on $[0,\pi]$, extended as each transform assumes", fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
"""Reconstruct from k coefficients, both ways."""
def dct_reconstruct(y, k):
    c = dct(y, type=2, norm="ortho")
    c[k:] = 0
    return idct(c, type=2, norm="ortho")

def dft_reconstruct(y, k):
    c = np.fft.rfft(y)
    c[k:] = 0
    return np.fft.irfft(c, n=len(y))

ks = [1, 2, 3, 4, 6]
fig, axes = plt.subplots(1, len(ks), figsize=(3.0 * len(ks), 3.0), sharey=True)
for ax, k in zip(axes, ks):
    ax.plot(x, y, color="black", lw=2.4, alpha=0.35, label="true")
    ax.plot(x, dft_reconstruct(y, k), color="#c05621", lw=1.7, label="DFT")
    ax.plot(x, dct_reconstruct(y, k), color="#1f6f8b", lw=1.7, label="DCT")
    ax.set_title(f"k = {k}", fontsize=10); ax.grid(alpha=0.25)
axes[0].legend(fontsize=8); axes[0].set_ylabel("y")
fig.suptitle("Reconstruction from the first k coefficients -- note the DFT's ringing near the ends", fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
"""Error against coefficient count -- the quantitative version."""
ks = np.arange(1, 33)
err_dct = [np.sqrt(np.mean((dct_reconstruct(y, k) - y) ** 2)) for k in ks]
err_dft = [np.sqrt(np.mean((dft_reconstruct(y, k) - y) ** 2)) for k in ks]

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogy(ks, err_dft, "o-", color="#c05621", ms=4, label="DFT")
ax.semilogy(ks, err_dct, "o-", color="#1f6f8b", ms=4, label="DCT")
ax.set_xlabel("coefficients retained"); ax.set_ylabel("RMSE (log scale)")
ax.set_title(r"DCT converges faster on $\sqrt{x}$ -- no boundary jump to represent")
ax.grid(alpha=0.3, which="both"); ax.legend(); fig.tight_layout(); plt.show()

for k in (2, 4, 8):
    print(f"k={k:>2}:  DCT RMSE {err_dct[k-1]:.4f}   DFT RMSE {err_dft[k-1]:.4f}"
          f"   ({err_dft[k-1] / err_dct[k-1]:.1f}x worse)")
print()
print("This is why JPEG uses the DCT. The same argument applies to every 8x8 block of an")
print("image: adjacent blocks rarely match at their edges, and a transform that did not")
print("handle that would spend its coefficients on artificial discontinuities.")

---
## 7. Building the network

The architecture from the paper, written out here rather than imported, so every choice is
visible. Three things drive it:

**The DC plane is special.** It is a real image - a coarse luma thumbnail - so it gets
convolutions of its own first, at the full 15x20 block grid.

**The AC and chroma planes are not.** Each is one number per block describing a specific
frequency. They carry no fine spatial structure worth convolving at full resolution, so
they are average-pooled to the trunk's resolution and concatenated as extra channels.

**Everything after the join is ordinary.** 3x3 convolutions, BatchNorm, ReLU, dropout,
global average pooling, and a linear classifier.

The channel widths are arguments, so they can be changed without touching anything else.

In [ ]:
"""The network, from scratch."""
import torch.nn.functional as F

def conv_bn_relu(in_ch, out_ch, stride=1, dropout=0.0):
    """3x3 convolution -> BatchNorm -> ReLU, optionally followed by dropout.

    Padding is 1 so a stride-1 block preserves the grid size. BatchNorm carries
    the bias, but the conv keeps one anyway: at export time BN is folded into
    the conv weights, and having the bias already there makes that fold a plain
    arithmetic identity."""
    layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
              nn.BatchNorm2d(out_ch),
              nn.ReLU(inplace=True)]
    if dropout:
        layers.append(nn.Dropout2d(dropout))
    return nn.Sequential(*layers)


class DctCnn(nn.Module):
    """Attribute names match the project's DctCnnClassifier deliberately -- the
    QAT initialiser copies weights across by name, so this model can be handed
    straight to the existing quantization and export path."""

    def __init__(self, cfg, lum_ch=16, stride2_ch=32, post_concat_ch=64,
                 extra_ch=(32,), dropout=0.3):
        super().__init__()
        self.num_ac_coeffs = cfg.num_ac_coeffs
        self.use_chroma = cfg.use_chroma

        # --- the DC branch: a real (if tiny) image ---
        self.lum_conv     = conv_bn_relu(1, lum_ch)                       # 15x20
        self.stride2_conv = conv_bn_relu(lum_ch, stride2_ch, stride=2)    # -> 8x10

        # --- the join: trunk + AC planes + chroma planes ---
        joined = stride2_ch + cfg.num_ac_coeffs
        if cfg.use_chroma:
            joined += 2 * cfg.num_chroma_coeffs
        self.post_concat_conv = conv_bn_relu(joined, post_concat_ch, dropout=dropout)

        # --- the rest of the trunk ---
        blocks, in_ch = [], post_concat_ch
        for out_ch in extra_ch:
            blocks += list(conv_bn_relu(in_ch, out_ch, dropout=dropout))
            in_ch = out_ch
        self.extra_convs = nn.Sequential(*blocks)

        # --- head ---
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(in_ch, cfg.num_classes)

    def forward_trunk(self, x, x_chroma=None):
        h = self.stride2_conv(self.lum_conv(x[:, 0:1]))     # DC plane only

        parts = [h]
        if self.num_ac_coeffs > 0:                          # AC planes, pooled to match
            parts.append(F.adaptive_avg_pool2d(x[:, 1:], output_size=h.shape[-2:]))
        if self.use_chroma:                                 # chroma planes, likewise
            parts.append(F.adaptive_avg_pool2d(x_chroma, output_size=h.shape[-2:]))
        h = torch.cat(parts, dim=1) if len(parts) > 1 else parts[0]

        h = self.extra_convs(self.post_concat_conv(h))
        return self.head_dropout(self.pool(h).flatten(1))

    def forward(self, x, x_chroma=None):
        return self.fc(self.forward_trunk(x, x_chroma))


LUM_CH, STRIDE2_CH, POST_CH, EXTRA_CH = 16, 32, 64, (32,)   # 
model = DctCnn(cfg, LUM_CH, STRIDE2_CH, POST_CH, EXTRA_CH, dropout=cfg.dropout)
print(model)

---
## 8. Model summary

Shapes and cost, before a single image is loaded.

In [ ]:
"""torchinfo summary at the configured input shapes."""
from torchinfo import summary
from dct_common.seeding import set_seed
from dct_common.device import get_device

set_seed(cfg.seed)
device = get_device()

dummy_y  = torch.zeros(1, cfg.num_coeffs, cfg.y_rows, cfg.y_cols)
dummy_c  = torch.zeros(1, 2 * cfg.num_chroma_coeffs, cfg.c_rows, cfg.c_cols)
print(summary(model, input_data=(dummy_y, dummy_c),
              col_names=("input_size", "output_size", "num_params", "mult_adds"),
              depth=2, verbose=0))

bn = sum(p.numel() for m in model.modules() if isinstance(m, nn.BatchNorm2d) for p in m.parameters())
tot = sum(p.numel() for p in model.parameters())
print(f"parameters as trained  : {tot:,}")
print(f"parameters as deployed : {tot - bn:,}   (BatchNorm folded into the convs at export)")

---
## 9. Splitting the data and training

`data/` is already split into train / val / test by class folder, so "splitting" here means
reading each split and turning JPEGs into coefficient planes.

The training loop is written out rather than imported - it is a plain AdamW loop with
cosine decay and best-on-validation checkpointing.

In [ ]:
"""Feature extraction. Reads data/, writes nothing."""
import shutil, tempfile
ensure_import_path()
from dct_common.splits import build_spatial_split

def load_split(split, limit=None):
    if limit is None:
        return build_spatial_split(DATA_DIR, split, list(CLASSES), cfg)
    tmp = Path(tempfile.mkdtemp(prefix=f"nb2_{split}_"))
    for c in CLASSES:
        (tmp / split / c).mkdir(parents=True, exist_ok=True)
        for p in sorted((DATA_DIR / split / c).glob("*.jpg"))[:limit]:
            os.symlink(p, tmp / split / c / p.name)
    try:
        X, Xc, y, paths = build_spatial_split(tmp, split, list(CLASSES), cfg)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    real = []
    for q in paths:
        try:
            real.append(str(DATA_DIR / Path(q).relative_to(tmp)))
        except ValueError:
            real.append(q)
    return X, Xc, y, real

val_limit = None if MAX_IMAGES_PER_CLASS is None else max(40, MAX_IMAGES_PER_CLASS // 4)
X_train, Xc_train, y_train, p_train = load_split("train", MAX_IMAGES_PER_CLASS)
X_val,   Xc_val,   y_val,   p_val   = load_split("val",   val_limit)
X_test,  Xc_test,  y_test,  p_test  = load_split("test",  val_limit)
for nm, X, y in (("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)):
    print(f"  {nm:<5} {X.shape}   class balance {np.bincount(y, minlength=len(CLASSES))}")

In [ ]:
"""Input calibration, then quantize-and-dequantize -- train on what export reconstructs."""
from dct_common.quantization import calibrate_channel_scales, quantize_planes

dc_scales     = calibrate_channel_scales(X_train[:, 0:1], cfg.calibration_percentile)
ac_scales     = calibrate_channel_scales(X_train[:, 1:], cfg.calibration_percentile) if cfg.num_ac_coeffs else np.zeros(0)
chroma_scales = calibrate_channel_scales(Xc_train, cfg.calibration_percentile)
print(f"dc {dc_scales[0]:.4f}   ac {np.array2string(ac_scales, precision=4)}   "
      f"chroma {np.array2string(chroma_scales, precision=4)}")

def qdq(X, Xc):
    q_dc = quantize_planes(X[:, 0:1], dc_scales)
    q_ac = (quantize_planes(X[:, 1:], ac_scales) if cfg.num_ac_coeffs
            else np.zeros((X.shape[0], 0, cfg.y_rows, cfg.y_cols), dtype=np.int8))
    q_ch = quantize_planes(Xc, chroma_scales)
    x_dq = np.concatenate([
        q_dc.astype(np.float32) * dc_scales.astype(np.float32)[None, :, None, None],
        q_ac.astype(np.float32) * (ac_scales.astype(np.float32)[None, :, None, None] if cfg.num_ac_coeffs else 1.0),
    ], axis=1)
    return q_dc, q_ac, q_ch, x_dq, q_ch.astype(np.float32) * chroma_scales.astype(np.float32)[None, :, None, None]

Q_dc_tr, Q_ac_tr, Q_ch_tr, Xtr, Xctr = qdq(X_train, Xc_train)
Q_dc_va, Q_ac_va, Q_ch_va, Xva, Xcva = qdq(X_val,   Xc_val)
Q_dc_te, Q_ac_te, Q_ch_te, Xte, Xcte = qdq(X_test,  Xc_test)

def loader(X, Xc, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(Xc), torch.from_numpy(y.astype(np.int64)))
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle, pin_memory=(device.type == "cuda"))

train_loader = loader(Xtr, Xctr, y_train, True)
val_loader   = loader(Xva, Xcva, y_val,   False)
test_loader  = loader(Xte, Xcte, y_test,  False)

In [ ]:
"""The training loop, written out."""
@torch.no_grad()
def evaluate(net, dl):
    net.eval()
    preds, labels, loss_sum = [], [], 0.0
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    for xb, xcb, yb in dl:
        xb, xcb, yb = xb.to(device), xcb.to(device), yb.to(device)
        out = net(xb, xcb)
        loss_sum += crit(out, yb).item() * xb.size(0)
        preds.append(out.argmax(1).cpu()); labels.append(yb.cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return {"loss": loss_sum / len(dl.dataset), "accuracy": float((preds == labels).mean()),
            "preds": preds, "labels": labels}

set_seed(cfg.seed)
model = DctCnn(cfg, LUM_CH, STRIDE2_CH, POST_CH, EXTRA_CH, dropout=cfg.dropout).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FLOAT_EPOCHS)
crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

best_acc, best_state = -1.0, None
for epoch in range(FLOAT_EPOCHS):
    model.train()
    running = 0.0
    for xb, xcb, yb in train_loader:
        xb, xcb, yb = xb.to(device), xcb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb, xcb), yb)
        loss.backward(); opt.step()
        running += loss.item() * xb.size(0)
    sched.step()
    va = evaluate(model, val_loader)
    flag = ""
    if va["accuracy"] > best_acc:
        best_acc, best_state = va["accuracy"], {k: v.detach().clone() for k, v in model.state_dict().items()}
        flag = "  <- best"
    print(f"epoch {epoch + 1:>3}/{FLOAT_EPOCHS}  train_loss {running / len(train_loader.dataset):.4f}"
          f"  val_acc {va['accuracy']:.4f}{flag}")

model.load_state_dict(best_state)
float_test = evaluate(model, test_loader)
print(f"\nfloat test accuracy: {float_test['accuracy']:.4f}")

---
## 10. Quantization-aware training: what and why

The ESP32 has no floating-point throughput worth using for this. Every weight and every
activation on the device is an **8-bit integer**, and the arithmetic is integer
multiply-accumulate with a fixed-point rescale between layers.

**Post-training quantization** - train in float, round the weights afterwards - loses
accuracy, because the network was never told that rounding was coming. Weights sitting
near a rounding boundary move, activations clip at the ends of their range, and nothing in
training pushed against either.

**Quantization-aware training** puts the rounding *inside* the training graph. Each forward
pass quantizes weights and activations to int8 and immediately dequantizes them, so the
network computes with values it can actually represent, while gradients flow through the
rounding as if it were the identity (the "straight-through estimator"). The network then
learns weights that survive rounding - it moves them away from boundaries and adapts to the
clipping.

Three details of this implementation:

- **Per-output-channel symmetric weights.** Each convolution filter gets its own scale, no
  zero point. A channel with a small dynamic range is not forced to share a scale with a
  large one.
- **Activations use an EMA of the observed range**, tracked during training, which becomes
  the fixed scale used on the device.
- **BatchNorm stays unfolded during QAT** and is folded into the convolution weights only
  at export. Folding early would mean training through a fold that changes every step.

Starting from the trained float model and fine-tuning for a few epochs at a low learning
rate is usually *better* than the float model it started from - the published run went
77.6% float to 81.8% after QAT.

In [ ]:
"""QAT: fine-tune the float model with fake quantization in the graph."""
from dct_common.models.cnn import DctCnnClassifierQat, init_cnn_qat_from_float

set_seed(cfg.seed)
qat_model = DctCnnClassifierQat(cfg).to(device)
init_cnn_qat_from_float(qat_model, model)      # copies by attribute name -- see DctCnn's docstring

qat_opt = torch.optim.AdamW(qat_model.parameters(), lr=1e-4, weight_decay=cfg.weight_decay)
best_acc, best_state = -1.0, None
for epoch in range(QAT_EPOCHS):
    qat_model.train()
    running = 0.0
    for xb, xcb, yb in train_loader:
        xb, xcb, yb = xb.to(device), xcb.to(device), yb.to(device)
        qat_opt.zero_grad()
        loss = crit(qat_model(xb, xcb), yb)
        loss.backward(); qat_opt.step()
        running += loss.item() * xb.size(0)
    va = evaluate(qat_model, val_loader)
    flag = ""
    if va["accuracy"] > best_acc:
        best_acc, best_state = va["accuracy"], {k: v.detach().clone() for k, v in qat_model.state_dict().items()}
        flag = "  <- best"
    print(f"qat epoch {epoch + 1:>3}/{QAT_EPOCHS}  train_loss {running / len(train_loader.dataset):.4f}"
          f"  val_acc {va['accuracy']:.4f}{flag}")

qat_model.load_state_dict(best_state)
qat_test = evaluate(qat_model, test_loader)
print(f"\nQAT test accuracy  : {qat_test['accuracy']:.4f}   (float was {float_test['accuracy']:.4f})")

---
## 11. int8 quantization

QAT *simulates* integer arithmetic in float. This step performs it: weights become int8
arrays, biases int32, and each layer's rescale becomes an integer multiplier and a right
shift. `Int8CnnReference` is pure NumPy and is the ground truth the C code must match
exactly.

Two numbers to watch: the int8 accuracy against the QAT accuracy, and the **agreement** -
how often the integer path predicts the same class as the float simulation it came from.

In [ ]:
"""Build the integer model."""
from dct_common.models.cnn import Int8CnnReference

model = model.to("cpu"); qat_model = qat_model.to("cpu")
int8_ref = Int8CnnReference(qat_model, float(dc_scales[0]), ac_scales, chroma_scales, cfg, train_loader)

int8_preds = int8_ref.predict(Q_dc_te, Q_ac_te, Q_ch_te)
int8_acc = int8_ref.accuracy(Q_dc_te, Q_ac_te, Q_ch_te, y_test)
agreement = float((int8_preds == qat_test["preds"]).mean())

print(f"float test accuracy : {float_test['accuracy']:.4f}")
print(f"QAT   test accuracy : {qat_test['accuracy']:.4f}")
print(f"int8  test accuracy : {int8_acc:.4f}")
print(f"agreement int8 vs QAT: {agreement:.4f}")
print()
print("Weight dtypes now on the way to the device:")
print(f"  lum_conv weights  {int8_ref.w1_q.dtype}  shape {int8_ref.w1_q.shape}")
print(f"  lum_conv bias     {int8_ref.b1_q.dtype}")
print(f"  rescale           multiplier {int8_ref.mult1.dtype}, shift {int8_ref.shift1.dtype}")

---
## 12. Export and verify

The exporter writes a self-contained C header - weights, biases, multipliers, shifts, and
the generated C for pooling and the dense layer.

Then the verifier compiles that header with a plain C compiler, runs it on real test
images, and checks the logits are **bit-identical** to the NumPy reference. Two independent
implementations of the same fixed-point arithmetic have to agree exactly before the header
is worth flashing.

In [ ]:
"""Save artifacts and export the C header -- all inside youtube/output/first_principles/."""
from dct_common.quantization import ragged_object_array
from export_cnn_c_weights import export_one

guard(NB_OUT)
np.savez(
    NB_OUT / "quantized_model.npz",
    lum_weight=int8_ref.w1_q, lum_bias=int8_ref.b1_q, lum_mult=int8_ref.mult1, lum_shift=int8_ref.shift1,
    stride2_weight=int8_ref.w2_q, stride2_bias=int8_ref.b2_q, stride2_mult=int8_ref.mult2, stride2_shift=int8_ref.shift2,
    post_concat_weight=int8_ref.w3_q, post_concat_bias=int8_ref.b3_q, post_concat_mult=int8_ref.mult3, post_concat_shift=int8_ref.shift3,
    extra_weights=ragged_object_array([w for w, _, _, _ in int8_ref.extra_layers_q]),
    extra_biases=ragged_object_array([b for _, b, _, _ in int8_ref.extra_layers_q]),
    extra_mults=ragged_object_array([m for _, _, m, _ in int8_ref.extra_layers_q]),
    extra_shifts=ragged_object_array([s for _, _, _, s in int8_ref.extra_layers_q]),
    output_weight=int8_ref.w_out_q, output_bias=int8_ref.b_out_q, output_mult=int8_ref.mult_out, output_shift=int8_ref.shift_out,
    dc_scale=dc_scales, ac_scales=ac_scales, chroma_scales=chroma_scales,
)

manifest = {
    "format_version": 1,
    "capture_width": cfg.capture_width, "capture_height": cfg.capture_height,
    "chroma_subsampling": cfg.chroma_subsampling,
    "y_rows": cfg.y_rows, "y_cols": cfg.y_cols, "c_rows": cfg.c_rows, "c_cols": cfg.c_cols,
    "num_ac_coeffs": cfg.num_ac_coeffs, "num_coeffs": cfg.num_coeffs,
    "num_chroma_ac_coeffs": cfg.num_chroma_ac_coeffs, "num_chroma_coeffs": cfg.num_chroma_coeffs,
    "coefficient_order": cfg.coeff_scan_order, "use_chroma": cfg.use_chroma,
    "lum_channels": LUM_CH, "stride2_channels": STRIDE2_CH,
    "post_concat_channels": POST_CH, "extra_conv_channels": list(EXTRA_CH),
    "num_classes": cfg.num_classes, "class_names": list(cfg.active_class_names),
    "act_scale_lum": int8_ref.act_scale_lum, "act_scale_stride2": int8_ref.act_scale_stride2,
    "act_scale_postconcat": int8_ref.act_scale_postconcat, "output_scale": int8_ref.output_scale,
    "float_test_accuracy": float_test["accuracy"], "qat_test_accuracy": qat_test["accuracy"],
    "int8_reference_test_accuracy": int8_acc, "int8_vs_qat_agreement": agreement,
    "notebook_demo": True,
}
(NB_OUT / "model_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

export_one(NB_OUT.name, NB_OUT)

In [ ]:
"""Verify the C header against the NumPy reference -- bit-exact or it fails."""
import verify_cnn_c_export as vcx

# The verifier resolves model directories under python_code/output by default.
# Point it at ours instead; nothing in the project tree is read for output.
vcx.OUTPUT_ROOT = NB_OUT.parent
ok = vcx.verify_one(NB_OUT.name)
print()
print("PASSED -- the exported C is bit-identical to the Python int8 reference." if ok
      else "FAILED -- do not flash this header.")

In [ ]:
"""Everything this notebook produced."""
for p in sorted(NB_OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(NB_DIR)}   ({p.stat().st_size / 1024:.0f} KB)")

---
## What this notebook argued

The DCT is not an implementation detail of JPEG that we happen to exploit. It is a
transform chosen because it concentrates a natural image's energy into a handful of
coefficients without inventing discontinuities - which is exactly the property that makes
those same coefficients a good classifier input.

The camera computes it in hardware, for free, before the microcontroller sees the frame.
Reading a few coefficients per block instead of decoding to pixels turns a 467 ms
classification into a 20 ms one on the same chip.